# EA2 — Siniestralidad Vial en Medellín

**Objetivo:** Realizar el proceso de limpieza y enriquecimiento del **dataset**.

## 1) Preparación e inspección rápida

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name.lower() == "notebooks" else NB_DIR

DATA_DIR  = PROJECT_ROOT / "data"
DOCS_DIR  = PROJECT_ROOT / "docs"
PLOTS_DIR = DOCS_DIR / "graficos"
DB_DIR    = PROJECT_ROOT / "db"
for d in [DATA_DIR, PLOTS_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "proyecto.db"
con = sqlite3.connect(str(DB_PATH))
cur = con.cursor()

print("Proyecto   :", PROJECT_ROOT)
print("Base SQLite:", DB_PATH)
print("Gráficos   :", PLOTS_DIR)


Proyecto   : c:\Dev\proyecto_integrado_ea1
Base SQLite: c:\Dev\proyecto_integrado_ea1\db\proyecto.db
Gráficos   : c:\Dev\proyecto_integrado_ea1\docs\graficos


## 2) Cargar desde SQLite y Normalización

In [2]:
def table_exists(conn, name):
    q = "SELECT name FROM sqlite_master WHERE type='table' AND lower(name)=lower(?);"
    return pd.read_sql(q, conn, params=[name]).shape[0] > 0

if table_exists(con, "incidentes"):
    df = pd.read_sql("SELECT * FROM incidentes", con)
    source_table = "incidentes"
else:
    
    assert table_exists(con, "incidentes_raw"), "No hay 'incidentes' ni 'incidentes_raw' en la base."
    df = pd.read_sql("SELECT * FROM incidentes_raw", con)
    source_table = "incidentes"
   
    def norm_col(c: str) -> str:
        c = c.strip().lower().replace(" ", "_")
        for a,b in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
            c = c.replace(a,b)
        while "__" in c: c = c.replace("__","_")
        return c
    df.columns = [norm_col(c) for c in df.columns]

print("Tabla origen:", source_table)
print("Shape bruto :", df.shape)
df.head(3)


Tabla origen: incidentes
Shape bruto : (270765, 19)


,id,fecha,hora,anio,mes,comuna_codigo,comuna,barrio,clase_accidente,gravedad,diseno,direccion,direccion_encasillada,x_magna,y_magna,cbml,expediente,nro_radicado,location
0,1,2015-10-21,10:58:00,2015,10,10,La Candelaria,Barrio Colón,Caida Ocupante,Con heridos,Tramo de via,CR 46 CL 43,CR 046 043 000 00000,834949.69,1182357.34,1013,A000259731,1508668,"[-75.5688011014, 6.24312304123]"
1,2,2015-11-05,08:00:00,2015,11,10,La Candelaria,San Diego,Choque,Solo daños,Tramo de via,CR 43 A CL 29,CR 043 A 029 000 00000,834880.17,1180762.02,1020,A000261725,1510621,"[-75.5693883283, 6.22870030622]"
2,3,2015-10-21,12:40:00,2015,10,2,Santa Cruz,La Francia,Otro,Con heridos,Tramo de via,CR 46 CL 37,CR 046 037 000 00000,837004.94,1188499.17,0205,A000259739,1508691,"[-75.5503911403, 6.29869502502]"


## 3) Limpieza

In [3]:
# Trim en texto
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan})

# Duplicados exactos
dup_before = df.duplicated().sum()
df = df.drop_duplicates()
# Duplicados por claves si existen
keys = [k for k in ["expediente","nro_radicado","fecha","fecha_accidentes","fecha_accidente"] if k in df.columns]
dup_keys = 0
if keys:
    dup_keys = df.duplicated(subset=keys).sum()
    df = df.drop_duplicates(subset=keys)

# Tipificación mínima
for c in ["numcomuna","x_magna","y_magna","x","y"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"Duplicados exactos eliminados : {dup_before}")
if keys: print(f"Duplicados por {keys} eliminados: {dup_keys}")
print("Nulos (top 10):\n", df.isna().sum().sort_values(ascending=False).head(10))
print("Shape limpio:", df.shape)



Duplicados exactos eliminados : 0
Duplicados por ['expediente', 'nro_radicado', 'fecha'] eliminados: 9
Nulos (top 10):
 id                 0
fecha              0
hora               0
anio               0
mes                0
comuna_codigo      0
comuna             0
barrio             0
clase_accidente    0
gravedad           0
dtype: int64
Shape limpio: (270756, 19)


## 4) Enriquecimiento

In [ ]:
fecha_series = None
hora_series  = None

if "fecha" in df.columns and df["fecha"].notna().any():
    fecha_series = pd.to_datetime(df["fecha"], errors="coerce")
    if "hora" in df.columns:
        hora_series = df["hora"]
else:
    for cand in ["fecha_accidentes","fecha_accidente"]:
        if cand in df.columns:
            dt = pd.to_datetime(df[cand], errors="coerce", infer_datetime_format=True)
            fecha_series = dt
            hora_series  = dt.dt.strftime("%H:%M:%S")
            break

if fecha_series is None:
    # Generar aleatoria para los que no tienen (2022-01-01..2024-12-31)
    rng = np.random.default_rng(42)
    start = np.datetime64("2022-01-01"); end = np.datetime64("2024-12-31")
    rand_days = rng.integers(0, (end - start).astype(int)+1, size=len(df))
    fecha_series = pd.to_datetime(start + rand_days.astype("timedelta64[D]"))

df["fecha"] = pd.to_datetime(fecha_series).dt.date
if hora_series is None:
    df["hora"] = None
else:
    df["hora"] = hora_series

# 2) Derivados de tiempo
fdt = pd.to_datetime(df["fecha"], errors="coerce")
df["anio"]     = fdt.dt.year
df["mes"]      = fdt.dt.month
df["dia"]      = fdt.dt.day
df["anio_mes"] = fdt.dt.to_period("M").astype(str)

# 3) Hora numérica y franja
def _to_hour(h):
    try: return int(str(h)[:2])
    except: return np.nan
df["hora_num"] = df["hora"].apply(_to_hour)

def franja(h):
    if pd.isna(h): return "Sin hora"
    h = int(h)
    if 0<=h<=5: return "Madrugada"
    if 6<=h<=11: return "Mañana"
    if 12<=h<=17: return "Tarde"
    return "Noche"
df["franja_horaria"]  = df["hora_num"].apply(franja)
df["es_fin_de_semana"] = pd.to_datetime(df["fecha"]).dt.dayofweek.isin([5,6])

# 4) Normalizaciones de variables clave
if "gravedad_accidente" in df.columns and "gravedad" not in df.columns:
    df["gravedad"] = df["gravedad_accidente"].astype(str).str.title()
if "clase_accidente" in df.columns:
    df["clase_accidente"] = df["clase_accidente"].astype(str).str.title()
for txtcol in ["comuna","barrio"]:
    if txtcol in df.columns:
        df[txtcol] = df[txtcol].astype(str).str.title()
if "numcomuna" in df.columns and "comuna_codigo" not in df.columns:
    df["comuna_codigo"] = pd.to_numeric(df["numcomuna"], errors="coerce").astype("Int64")

print("Enriquecimiento OK. Columnas clave presentes:",
      [c for c in ["fecha","anio","mes","dia","anio_mes","hora","hora_num","franja_horaria",
                   "clase_accidente","gravedad","comuna","comuna_codigo"] if c in df.columns])


Enriquecimiento OK. Columnas clave presentes: ['fecha', 'anio', 'mes', 'dia', 'anio_mes', 'hora', 'hora_num', 'franja_horaria', 'clase_accidente', 'gravedad', 'comuna', 'comuna_codigo']


## 5) Exportación desde la base

In [5]:
OUT_CSV = DATA_DIR / "dataset_enriquecido.csv"
df.to_csv(OUT_CSV, index=False, encoding="utf-8")

# Guardar en SQLite (reemplaza si ya existía)
df.to_sql("incidentes_enriquecido", con, if_exists="replace", index=False)

# Índices útiles para la tabla enriquecida
cur.executescript("""
CREATE INDEX IF NOT EXISTS idx_inc_enr_fecha   ON incidentes_enriquecido(fecha);
CREATE INDEX IF NOT EXISTS idx_inc_enr_comuna  ON incidentes_enriquecido(comuna);
CREATE INDEX IF NOT EXISTS idx_inc_enr_clase   ON incidentes_enriquecido(clase_accidente);
""")
con.commit()

print("CSV enriquecido →", OUT_CSV)
print("Tabla SQLite    → incidentes_enriquecido (reemplazada)")


CSV enriquecido → c:\Dev\proyecto_integrado_ea1\data\dataset_enriquecido.csv
Tabla SQLite    → incidentes_enriquecido (reemplazada)


## 6) Descriptivos y visualizaciones

**Archivos generados:** `docs/graficos/`

In [6]:
from IPython.display import display

# === Stats rápidas
num_cols = [c for c in ["anio","mes","dia","hora_num","comuna_codigo"] if c in df.columns]
print("Describe (numéricas):")
if num_cols:
    display(df[num_cols].describe())

# === Gráficos: usa solamente matplotlib
plt.close("all")

# 1) Top 10 comunas
if "comuna" in df.columns:
    topc = df["comuna"].value_counts().head(10)
    plt.figure(figsize=(8,4)); topc.plot(kind="bar")
    plt.title("Top 10 comunas por incidentes"); plt.xlabel("Comuna"); plt.ylabel("Total")
    plt.tight_layout(); f1 = PLOTS_DIR / "01_top10_comunas.png"
    plt.savefig(f1, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f1)

# 2) Clase de accidente
if "clase_accidente" in df.columns:
    vc = df["clase_accidente"].value_counts()
    plt.figure(figsize=(8,4)); vc.plot(kind="bar")
    plt.title("Distribución por clase de accidente"); plt.xlabel("Clase"); plt.ylabel("Total")
    plt.tight_layout(); f2 = PLOTS_DIR / "02_clase_accidente.png"
    plt.savefig(f2, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f2)

# 3) Gravedad
if "gravedad" in df.columns:
    vg = df["gravedad"].value_counts()
    plt.figure(figsize=(6,4)); vg.plot(kind="bar")
    plt.title("Distribución por gravedad"); plt.xlabel("Gravedad"); plt.ylabel("Total")
    plt.tight_layout(); f3 = PLOTS_DIR / "03_gravedad.png"
    plt.savefig(f3, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f3)

# 4) Serie temporal mensual
if "anio_mes" in df.columns:
    ts = df.groupby("anio_mes").size().sort_index()
    plt.figure(figsize=(9,4)); ts.plot()
    plt.title("Incidentes por mes (serie temporal)"); plt.xlabel("Año-Mes"); plt.ylabel("Total")
    plt.xticks(rotation=45); plt.tight_layout(); f4 = PLOTS_DIR / "04_temporal_mensual.png"
    plt.savefig(f4, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f4)

# 5) Histograma por hora
if "hora_num" in df.columns:
    plt.figure(figsize=(7,4)); df["hora_num"].dropna().astype(int).plot(kind="hist", bins=24)
    plt.title("Histograma por hora del día"); plt.xlabel("Hora"); plt.ylabel("Frecuencia")
    plt.tight_layout(); f5 = PLOTS_DIR / "05_hist_hora.png"
    plt.savefig(f5, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f5)

# 6) Correlación simple (si hay ≥2 numéricas)
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(6,5))
    plt.imshow(corr, interpolation="nearest"); plt.colorbar()
    plt.xticks(range(len(num_cols)), num_cols, rotation=45); plt.yticks(range(len(num_cols)), num_cols)
    plt.title("Correlación (variables numéricas)")
    plt.tight_layout(); f6 = PLOTS_DIR / "06_correlacion.png"
    plt.savefig(f6, dpi=150, bbox_inches="tight"); plt.close(); print("✔", f6)


Describe (numéricas):


,anio,mes,dia,hora_num
count,270756.000000,270756.000000,270756.00000,270756.000000
mean,2016.953730,6.685617,15.69187,13.694227
std,1.758066,3.414359,8.72458,6.840564
min,2014.000000,1.000000,1.00000,0.000000
25%,2015.000000,4.000000,8.00000,10.000000
50%,2017.000000,7.000000,16.00000,15.000000
75%,2018.000000,10.000000,23.00000,19.000000
max,2020.000000,12.000000,31.00000,23.000000


✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\01_top10_comunas.png
✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\02_clase_accidente.png
✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\03_gravedad.png
✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\04_temporal_mensual.png
✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\05_hist_hora.png
✔ c:\Dev\proyecto_integrado_ea1\docs\graficos\06_correlacion.png
